In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import pytz

def show_time_series_chart(csv_file_path):
    # 1. Check Current Time in IST
    ist_tz = pytz.timezone('Asia/Kolkata')
    current_time_ist = datetime.now(ist_tz)
    
    # 2. Time-Gate Logic: Only execute between 18:00 (6 PM) and 21:00 (9 PM)
    if not (18 <= current_time_ist.hour < 21):
        print(f"Graph is currently hidden. It is only available between 6 PM and 9 PM IST. (Current time: {current_time_ist.strftime('%I:%M %p')} IST)")
        return  # Exit the function

    # 3. Load Data
    df = pd.read_csv(csv_file_path)

    # 4. Clean Data
    # Convert 'Reviews' to numeric
    df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')
    
    # Convert 'Installs' to numeric
    df['Installs'] = df['Installs'].astype(str).str.replace(r'[+,]', '', regex=True)
    df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')
    
    # Convert 'Last Updated' to datetime
    df['Last Updated'] = pd.to_datetime(df['Last Updated'], errors='coerce')

    # 5. Apply Complex Filters
    # App name must not start with X, Y, or Z (case-insensitive)
    df = df[~df['App'].str.lower().str.startswith(('x', 'y', 'z'), na=False)]
    
    # App name must not contain the letter 'S' (case-insensitive)
    df = df[~df['App'].str.lower().str.contains('s', na=False)]
    
    # Category must start with E, C, or B
    df = df[df['Category'].str.startswith(('E', 'C', 'B'), na=False)]
    
    # Reviews must be > 500
    df = df[df['Reviews'] > 500]

    # 6. Apply Category Translations
    # Note: Since we filtered for categories starting with E, C, or B, 'DATING' will 
    # technically be excluded by the filter above. However, the translation logic is included 
    # as requested in case filter parameters change in the future.
    translations = {
        'BEAUTY': 'सौंदर्य',          # Hindi
        'BUSINESS': 'வணிகம்',         # Tamil
        'DATING': 'Partnersuche'      # German
    }
    df['Category'] = df['Category'].replace(translations)

    # 7. Aggregate Data Over Time (Monthly)
    df = df.dropna(subset=['Last Updated', 'Installs'])
    df['Month'] = df['Last Updated'].dt.to_period('M')
    
    # Group by translated Category and Month
    time_series_df = df.groupby(['Category', 'Month'])['Installs'].sum().reset_index()
    
    # Convert period back to timestamp for plotting
    time_series_df['Month'] = time_series_df['Month'].dt.to_timestamp()

    # 8. Plot the Time Series Chart
    fig, ax = plt.subplots(figsize=(14, 7))
    categories = time_series_df['Category'].unique()

    for category in categories:
        cat_data = time_series_df[time_series_df['Category'] == category].sort_values('Month')
        
        # Calculate Month-over-Month (MoM) growth percentage
        cat_data['MoM_Growth'] = cat_data['Installs'].pct_change()
        
        # Plot the main trend line
        ax.plot(cat_data['Month'], cat_data['Installs'], marker='o', label=category)
        
        # Highlight periods with > 20% MoM growth
        # We use fill_between to shade the area under the curve for these specific periods
        growth_periods = cat_data['MoM_Growth'] > 0.20
        
        ax.fill_between(
            cat_data['Month'], 
            0, 
            cat_data['Installs'], 
            where=growth_periods, 
            alpha=0.3, 
            step='mid',
            label=f'{category} (>20% Growth)' if growth_periods.any() else ""
        )

    # Formatting the plot
    ax.set_title('Total Installs Over Time by Category (Filtered)', fontsize=14)
    ax.set_xlabel('Date (Last Updated)', fontsize=12)
    ax.set_ylabel('Total Installs', fontsize=12)
    
    # Handle legends carefully to avoid duplicate labels from fill_between
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles)) # Removes duplicates
    ax.legend(by_label.values(), by_label.keys(), loc='upper left', bbox_to_anchor=(1, 1))

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Usage
show_time_series_chart('googleplaystore.csv')

Graph is currently hidden. It is only available between 6 PM and 9 PM IST. (Current time: 02:18 PM IST)
